# Child fall risk: Colab runner
Runtime → Change runtime type → **GPU**. Run cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put the repo folder at MyDrive/childfall (or git clone it)
%cd /content/drive/MyDrive/childfall
!pip -q install -r requirements.txt

## 0. Smoke test on synthetic data (2 min)

In [ ]:
!python scripts/make_synthetic.py --out data_synth --n 300
!python scripts/train.py --data data_synth --model stgcn --epochs 5 --out runs_synth
!python scripts/evaluate.py --data data_synth --ckpts runs_synth/*/best.pt --out results_synth --conditions clean occlude_20pct

## 1. Pose extraction
Put videos in `data/videos/` (public datasets + team recordings).

In [ ]:
!python scripts/extract_pose.py --videos data/videos --out data/poses --model yolo11s-pose.pt

## 2. Labels
Create `data/labels.csv` (clip_id,label,subject,source,split) using docs/labeling_guide.md. Split **by subject**.

In [ ]:
import pandas as pd
lab = pd.read_csv('data/labels.csv')
print(lab.groupby(['split','label']).size().unstack(fill_value=0))
assert not set(lab[lab.split=='train'].subject) & set(lab[lab.split=='test'].subject), 'subject leak between train and test'

## 3. Train (3 seeds each)

In [ ]:
for m in ['stgcn','lstm']:
    for s in [0,1,2]:
        !python scripts/train.py --data data --model {m} --seed {s} --epochs 60 --out runs

## 4. Evaluate: clean + robustness

In [ ]:
!python scripts/evaluate.py --data data --ckpts runs/*/best.pt --out results
from IPython.display import Markdown
Markdown(open('results/eval_test.md').read())

## 5. Streaming: false alarms per hour

In [ ]:
!python scripts/evaluate_stream.py --root data/stream --rules
!python scripts/evaluate_stream.py --root data/stream --ckpt runs/stgcn_seed0/best.pt

## 6. Skeleton-only demo video

In [ ]:
!python scripts/demo.py --source data/demo/demo.mp4 --ckpt runs/stgcn_seed0/best.pt --out demo_out.mp4